# Chapter 8: The Transformer Architecture

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch08_the_transformer_architecture.ipynb)


In [1]:
# Seeded before anything below runs, including the chapter's own examples.
#
# Those examples call torch.randn without a seed. This notebook is committed
# with its output stored, and a stored number that changes on every rebuild is
# noise printed as a result. Seeding here makes the whole file reproducible.
#
# Same seed as tools/claim_instances.py, which produced the slide numbers.
import torch
torch.manual_seed(20260729)
print('seeded', 20260729)

seeded 20260729


### 8.2.5 Numerical Walkthrough


In [2]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention.

    The `mask` argument uses the boolean convention (1 = allowed,
    0 = blocked), which differs from the additive {0, -inf} mask
    in Equation (line 92). Both conventions encode the same causal
    constraint; we use the boolean form here because PyTorch's
    `masked_fill` expects it.
    """
    d_k = Q.size(-1)
    # Compute attention scores: QK^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    # Apply causal mask if provided (mask==0 -> blocked, set to -inf)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    # Softmax to get attention weights
    weights = F.softmax(scores, dim=-1)
    # Weighted sum of values
    output = torch.matmul(weights, V)
    return output, weights

# Example: 1 batch, 4 positions, d_k = 8
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")      # [1, 4, 8]
print(f"Attention weights:\n{weights[0]}")   # [4, 4] matrix


Output shape: torch.Size([1, 4, 8])
Attention weights:
tensor([[0.0757, 0.2382, 0.1667, 0.5195],
        [0.1446, 0.0846, 0.0694, 0.7014],
        [0.1855, 0.1029, 0.6990, 0.0126],
        [0.1092, 0.1079, 0.7381, 0.0448]])


### 8.3.3 Parallel Attention and Concatenation

![Figure 8.2 -- Multi-Head Attention](../figures/fig-08-2.pdf)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.h = h
        self.d_k = d_model // h
        # Single projection matrices for all heads (batched)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X, mask=None):
        B, T, d_model = X.shape
        # Project and reshape into h heads: (B, T, d_model) -> (B, h, T, d_k)
        Q = self.W_Q(X).view(B, T, self.h, self.d_k).transpose(1, 2)
        K = self.W_K(X).view(B, T, self.h, self.d_k).transpose(1, 2)
        V = self.W_V(X).view(B, T, self.h, self.d_k).transpose(1, 2)
        # Scaled dot-product attention per head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        head_out = torch.matmul(weights, V)          # (B, h, T, d_k)
        # Concatenate heads and project
        out = head_out.transpose(1, 2).reshape(B, T, d_model)
        return self.W_O(out), weights

mha = MultiHeadAttention(d_model=16, h=4)
X = torch.randn(1, 6, 16)
output, weights = mha(X)
print(f"Output shape: {output.shape}")           # [1, 6, 16]
print(f"Head attention shapes: {weights.shape}") # [1, 4, 6, 6]


Output shape: torch.Size([1, 6, 16])
Head attention shapes: torch.Size([1, 4, 6, 6])


### 8.4.2 Sinusoidal Position Encodings


In [4]:
import torch
import math

def sinusoidal_positional_encoding(T, d_model):
    """Compute sinusoidal positional encodings for T positions and d_model dims.

    Requires d_model to be even (holds for every Transformer config in
    practice, since d_model is always a multiple of the head count h).
    """
    assert d_model % 2 == 0, "d_model must be even for sinusoidal encoding"
    PE = torch.zeros(T, d_model)
    pos = torch.arange(0, T, dtype=torch.float).unsqueeze(1)  # (T, 1)
    # Frequencies: 1/10000^(2i/d_model) for i = 0, 1, ..., d_model//2 - 1
    i = torch.arange(0, d_model // 2, dtype=torch.float)
    freq = torch.exp(-math.log(10000.0) * 2 * i / d_model)    # (d_model//2,)
    # Even dimensions: sin; odd dimensions: cos
    PE[:, 0::2] = torch.sin(pos * freq)
    PE[:, 1::2] = torch.cos(pos * freq)
    return PE

PE = sinusoidal_positional_encoding(T=128, d_model=64)
print(f"PE shape: {PE.shape}")                    # [128, 64]
print(f"Position 0 norm: {PE[0].norm():.4f}")
print(f"Position 127 norm: {PE[127].norm():.4f}")


PE shape: torch.Size([128, 64])
Position 0 norm: 5.6569
Position 127 norm: 5.6569


### 8.5.5 Putting It All Together: The Transformer Block


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, h)   # from Code Example 2
        self.norm2 = nn.LayerNorm(d_model)
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)

    def forward(self, x, mask=None):
        # Sublayer 1: Pre-LN self-attention with residual
        attn_out, _ = self.attn(self.norm1(x), mask=mask)
        x = x + attn_out
        # Sublayer 2: Pre-LN feed-forward with residual
        ff_out = self.ff2(F.relu(self.ff1(self.norm2(x))))
        x = x + ff_out
        return x

block = TransformerBlock(d_model=32, h=4, d_ff=128)
x = torch.randn(1, 5, 32)              # (batch, seq_len, d_model)
out = block(x)
print(f"Input shape:  {x.shape}")      # [1, 5, 32]
print(f"Output shape: {out.shape}")    # [1, 5, 32]  -- shape preserved


Input shape:  torch.Size([1, 5, 32])
Output shape: torch.Size([1, 5, 32])


---

## Summary

This notebook demonstrated the key code examples from Chapter 8: The Transformer Architecture. For the full mathematical exposition and discussion, refer to the textbook chapter.


---

## The results this chapter names, and the numbers behind them

Chapter 8 states twelve numbered results. The ones with a numeric instance are
below, and **these cells are the lecture's own code**: they are lifted verbatim
from `tools/claim_instances.py`, which is what generated every number on the
slides. Running a cell here reproduces exactly what you saw, because it is the
same function.

Each cell prints its result and each output is stored in this file, so the
notebook is readable without running anything and checkable by running it.

Change a constant and rerun: that is the exercise. `claim_8_2_scaling` with
`d_k=4` instead of 64, or `claim_8_4_equivariance` with a different permutation,
are the two that repay it most.


In [6]:
# Lifted from tools/claim_instances.py, which generated every number in
# the chapter 8 slides. Do not retype these: rerun
#     python tools/build_notebook.py ch08
# and the notebook picks up whatever the module now says.
import math
import torch

SEED = 20260729
D_MODEL = 512
N_HEADS = 8
D_FF = 2048
D_GPT2 = 768
H_GPT2 = 12

def tex_int(n):
    """Format an integer the way the deck does: 1{,}048{,}576."""
    return "{,}".join(reversed([str(n)[max(0, i - 3):i]
                                for i in range(len(str(n)), 0, -3)]))

def _bytes_human(n):
    for unit, size in (("GiB", 1 << 30), ("MiB", 1 << 20), ("KiB", 1 << 10)):
        if n >= size:
            return "%g %s" % (round(n / size, 3), unit)
    return "%d B" % n

def _grad_norm_linear_residual(d, depth, scale):
    """The exploding half of the boundary: a residual stack with no saturation.

    y = x + Wx repeated, so dy/dx = (I + W)^depth and the norm compounds. This is
    the regime layer normalisation exists for, and it is the honest counterpart
    to the vanishing measurement above.
    """
    torch.manual_seed(SEED)
    sub = torch.nn.Linear(d, d, bias=False)
    with torch.no_grad():
        sub.weight.mul_(scale)
    torch.manual_seed(SEED)
    v = torch.randn(d, requires_grad=True)
    h = v
    for _ in range(depth):
        h = h + sub(h)
    h.sum().backward()
    return float(v.grad.norm())

print('torch', torch.__version__, ' seed', SEED)

torch 2.13.0+cpu  seed 20260729


### Result 8.1, traced: attention is a weighted average of value vectors

In [7]:
def claim_8_1_walkthrough():
    """Attention on three tokens, end to end, in integers.

    Warrant traced, so every intermediate has to be followable by hand: the
    projections are small integer matrices, so Q, K, V and the scores are exact
    integers and the only inexact step is the softmax. This is the walkthrough
    that section 8.2.5 was deferred to the notebook for, and step 8c runs these
    identical numbers in a cell whose output is stored.
    """
    X = torch.tensor([[1., 0., 1., 0.],
                      [0., 2., 0., 2.],
                      [1., 1., 1., 1.]])
    Wq = torch.tensor([[1., 0., 1.], [1., 0., 0.], [0., 0., 1.], [0., 1., 1.]])
    Wk = torch.tensor([[0., 0., 1.], [1., 1., 0.], [0., 1., 0.], [1., 1., 0.]])
    Wv = torch.tensor([[0., 2., 0.], [0., 3., 0.], [1., 0., 3.], [1., 1., 0.]])

    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = Q.shape[1]
    S = Q @ K.T
    S_scaled = S / math.sqrt(d_k)
    A = torch.softmax(S_scaled, dim=-1)
    Z = A @ V

    #% The point of the small case, made numerically: row 0 of the output is a
    #% convex combination of the three value rows and nothing else. If this gap
    #% were not zero the slide's sentence "a weighted average of value vectors"
    #% would be decoration.
    recombined = sum(A[0, j] * V[j] for j in range(3))
    gap = float((Z[0] - recombined).abs().max())

    return {
        "d_model": 4, "d_k": d_k, "n_tokens": 3,
        "X": X.int().tolist(),
        "Wq": Wq.int().tolist(), "Wk": Wk.int().tolist(), "Wv": Wv.int().tolist(),
        "Q": Q.int().tolist(), "K": K.int().tolist(), "V": V.int().tolist(),
        "scores_unscaled": S.int().tolist(),
        "scores_scaled": [[round(float(x), 3) for x in r] for r in S_scaled],
        "attention_weights": [[round(float(x), 4) for x in r] for r in A],
        "output": [[round(float(x), 3) for x in r] for r in Z],
        "row_sums": [round(float(x), 6) for x in A.sum(-1)],
        "weighted_average_gap": gap,
        "_ok": gap < 1e-6 and abs(float(A.sum(-1).min()) - 1) < 1e-6,
    }

r = claim_8_1_walkthrough()
r.pop('_ok', None)

print('attention weights, row by row:')
for row in r['attention_weights']:
    print('  ', row)
print('row sums          ', r['row_sums'])
print('output            ', r['output'])
print('recombination gap ', r['weighted_average_gap'])

attention weights, row by row:
   [0.1361, 0.4319, 0.4319]
   [0.0009, 0.9088, 0.0903]
   [0.0074, 0.7547, 0.2378]
row sums           [1.0, 1.0, 1.0]
output             [[1.864, 6.319, 1.704], [1.999, 7.814, 0.273], [1.993, 7.48, 0.736]]
recombination gap  0.0


### Result 8.2, derived: unscaled logits have spread sqrt(d_k)

In [8]:
def claim_8_2_scaling(d_k=64, n_keys=10, n_estimate=4000):
    """Softmax saturation with and without the 1/sqrt(d_k) factor.

    Two sample sizes on purpose, and the difference between them is itself the
    lesson. The textbook prediction sd = sqrt(d_k) is a statement about q and k
    both being random. Once q is fixed, which is what happens at every position
    of every real forward pass, the conditional spread is |q|, and |q| is only
    approximately sqrt(d_k). Estimating that spread from the ten keys that fit on
    a slide gives an answer wrong by half, because the sample standard deviation
    of ten draws is itself noisy. So: ten keys to display, four thousand to
    estimate, and both numbers reported.
    """
    torch.manual_seed(SEED)
    q = torch.randn(d_k)
    K = torch.randn(n_keys, d_k)
    raw = K @ q
    scaled = raw / math.sqrt(d_k)
    #% Twenty independent streams, not one long one. A single sample of 4000 came
    #% out 3.7 standard errors high while it was being written, which looked like
    #% a defect in the theory and was a defect in the measurement. The exercise
    #% sheets tell students to report the spread over runs rather than one mean;
    #% the tool that generates the teaching material should do the same.
    reps = [float((torch.randn(n_estimate, d_k,
                               generator=torch.Generator().manual_seed(SEED + r))
                   @ q).std()) for r in range(20)]
    est_mean = sum(reps) / len(reps)
    est_sd = (sum((x - est_mean) ** 2 for x in reps) / len(reps)) ** 0.5

    p_raw = torch.softmax(raw, dim=0)
    p_scaled = torch.softmax(scaled, dim=0)

    #% The gradient of softmax vanishes as it saturates. Entropy is the cheapest
    #% honest summary of that, and it is what the slide should plot.
    def entropy(p):
        return float(-(p * p.clamp_min(1e-12).log()).sum())

    return {
        "d_k": d_k,
        "n_keys": n_keys,
        "n_estimate": n_estimate,
        "logit_sd_predicted_marginal": math.sqrt(d_k),
        "logit_sd_predicted_given_q": float(q.norm()),
        "logit_sd_estimated_mean": round(est_mean, 3),
        "logit_sd_estimated_sd": round(est_sd, 3),
        "n_replications": len(reps),
        "logit_sd_from_display_sample": float(raw.std()),
        "raw_logits": [round(float(x), 2) for x in raw],
        "scaled_logits": [round(float(x), 2) for x in scaled],
        "p_raw": [round(float(x), 4) for x in p_raw],
        "p_scaled": [round(float(x), 4) for x in p_scaled],
        "max_p_raw": round(float(p_raw.max()), 4),
        "max_p_scaled": round(float(p_scaled.max()), 4),
        "entropy_raw_nats": round(entropy(p_raw), 3),
        "entropy_scaled_nats": round(entropy(p_scaled), 3),
        "entropy_uniform_nats": round(math.log(n_keys), 3),
        "_ok": float(p_raw.max()) > 0.9 and float(p_scaled.max()) < 0.9,
    }

r = claim_8_2_scaling()
r.pop('_ok', None)

print('sd predicted marginally sqrt(d_k) =', round(r['logit_sd_predicted_marginal'], 2))
print('sd predicted for this q      |q| =', round(r['logit_sd_predicted_given_q'], 2))
print('sd measured over', r['n_replications'], 'runs   =', r['logit_sd_estimated_mean'], '+/-', r['logit_sd_estimated_sd'])
print('largest weight  unscaled', r['max_p_raw'], '  scaled', r['max_p_scaled'])
print('entropy (nats)  unscaled', r['entropy_raw_nats'], '  scaled', r['entropy_scaled_nats'])

sd predicted marginally sqrt(d_k) = 8.0
sd predicted for this q      |q| = 8.79
sd measured over 20 runs   = 8.784 +/- 0.133
largest weight  unscaled 0.9913   scaled 0.4616
entropy (nats)  unscaled 0.05   scaled 1.524


### Result 8.3, derived: attention is quadratic in sequence length

In [9]:
def claim_8_3_cost():
    """The quadratic term, and the width-dependent threshold where it bites.

    Warrant derived, and the instance is the cost table on the frame. Both
    columns of it are recomputed here: the ratio 4Td^2 : 2T^2d, which reduces to
    T : 2d, and the fp16 score matrix held for the backward pass. The third
    column is the frame's whole argument, so if 384 GiB ever stops being what
    the arithmetic says, the gate should go red before the room does.
    """
    rows = []
    for T in (512, 4096, 131072):
        proj_flops = 4 * T * D_GPT2 ** 2
        attn_flops = 2 * T ** 2 * D_GPT2
        #% One layer, one sequence, every head's score matrix, two bytes each.
        score_bytes = T * T * H_GPT2 * 2
        rows.append({
            "T": T,
            "T_tex": tex_int(T),
            "projection_flops": proj_flops,
            "attention_flops": attn_flops,
            "ratio": round(attn_flops / proj_flops, 2),
            "ratio_closed_form": round(T / (2 * D_GPT2), 2),
            "score_matrix_bytes": score_bytes,
            "score_matrix_human": _bytes_human(score_bytes),
        })

    #% The mistake the formula frame heads off, in one number: T^2 scalars, not T.
    scores_at_512 = 512 ** 2

    return {
        "d_model": D_GPT2, "n_heads": H_GPT2, "dtype_bytes": 2,
        "crossover_T": 2 * D_GPT2,
        "rows": rows,
        "scores_at_T_512": scores_at_512,
        "scores_at_T_512_tex": tex_int(scores_at_512),
        "_ok": (all(abs(r["ratio"] - r["ratio_closed_form"]) < 0.01
                    for r in rows)
                and rows[-1]["score_matrix_human"] == "384 GiB"),
    }

r = claim_8_3_cost()
r.pop('_ok', None)

print('crossover at T =', r['crossover_T'], '= 2d')
for row in r['rows']:
    print('  T = %7d   attention/projections %6.2f   scores %s' % (row['T'], row['ratio'], row['score_matrix_human']))

crossover at T = 1536 = 2d
  T =     512   attention/projections   0.33   scores 6 MiB
  T =    4096   attention/projections   2.67   scores 384 MiB
  T =  131072   attention/projections  85.33   scores 384 GiB


### Result 8.4, derived: attention cannot see word order

In [10]:
def claim_8_4_equivariance(n=4, d=16, d_k=8):
    """Permute the input rows; the output rows permute with them, exactly."""
    #% THE SEED IS SEARCHED FOR, DETERMINISTICALLY, AND HERE IS WHY.
    #%
    #% Equivariance holds for any weights, so the first version took the first
    #% draw and shipped it. Three of its four output rows were identical to two
    #% decimal places, because at full scale the logits over four keys saturate
    #% and every query lands on the same key. Every numerical check passed: the
    #% identity is exact whether or not the rows differ. The figure built from it
    #% showed a permutation rearranging rows a reader cannot tell apart, which
    #% demonstrates nothing.
    #%
    #% Scaling the projections down does not fix it, it swaps the failure: small
    #% logits give near-uniform attention, every row becomes the mean of V, and
    #% the rows collapse again. Both ends of the scale are degenerate and the
    #% usable region is in between, so the honest thing is to state the property
    #% the instance needs and search for a draw that has it, rather than to tune
    #% a constant until a picture looks right.
    #%
    #% First seed whose four output rows are pairwise separated by more than
    #% MIN_SEP. Deterministic, reported in the artefact, and re-searched from
    #% scratch on every run, so this cannot silently drift.
    MIN_SEP = 0.5
    chosen = None
    for offset in range(200):
        torch.manual_seed(SEED + offset)
        X = torch.randn(n, d)
        Wq, Wk, Wv = (torch.randn(d, d_k) * 0.5 for _ in range(3))
        A = torch.softmax((X @ Wq) @ (X @ Wk).T / math.sqrt(d_k), dim=-1)
        Z = A @ (X @ Wv)
        sep = float(min((Z[i] - Z[j]).abs().max()
                        for i in range(n) for j in range(i + 1, n)))
        if sep > MIN_SEP:
            chosen = (SEED + offset, sep)
            break

    def attend(x):
        Q, K, V = x @ Wq, x @ Wk, x @ Wv
        A = torch.softmax(Q @ K.T / math.sqrt(d_k), dim=-1)
        return A @ V

    perm = torch.tensor([2, 0, 3, 1])
    out = attend(X)
    out_perm = attend(X[perm])
    gap = float((out_perm - out[perm]).abs().max())

    #% The same test with a causal mask, which is the boundary of the claim: the
    #% mask makes position i attend to a set of size i, so it carries order
    #% information and the symmetry is gone.
    def attend_causal(x):
        Q, K, V = x @ Wq, x @ Wk, x @ Wv
        S = Q @ K.T / math.sqrt(d_k)
        S = S.masked_fill(torch.triu(torch.ones(n, n, dtype=torch.bool), 1),
                          float("-inf"))
        return torch.softmax(S, dim=-1) @ V

    gap_causal = float((attend_causal(X[perm]) - attend_causal(X)[perm])
                       .abs().max())

    return {
        "n_positions": n,
        "d_model": d,
        "d_k": d_k,
        "permutation": perm.tolist(),
        "max_abs_gap_unmasked": gap,
        "max_abs_gap_causal": round(gap_causal, 4),
        "output_row0_original": [round(float(x), 3) for x in out[0][:5]],
        "output_row0_permuted": [round(float(x), 3) for x in out_perm[0][:5]],
        #% The full output, both ways round, first five dimensions of each row.
        #% The figure for this claim shows that the permuted run produces the
        #% same rows in the permuted order, which is a statement about rows and
        #% cannot be made with row 0 alone.
        "output_original": [[round(float(v), 2) for v in r[:5]] for r in out],
        "output_permuted": [[round(float(v), 2) for v in r[:5]]
                            for r in out_perm],
        "seed_used": chosen[0] if chosen else None,
        "seed_offsets_tried": (chosen[0] - SEED + 1) if chosen else 200,
        "min_row_separation": round(chosen[1], 3) if chosen else None,
        "min_row_separation_required": MIN_SEP,
        #% The search has to have succeeded, as well as the identity holding.
        #% A degenerate instance passes every numerical check and still teaches
        #% the wrong thing, so it is a failure here.
        "_ok": (gap < 1e-5 and gap_causal > 1e-3 and chosen is not None),
    }

r = claim_8_4_equivariance()
r.pop('_ok', None)

print('permutation           ', [p + 1 for p in r['permutation']])
print('seed used             ', r['seed_used'])
print('no mask, max |gap|    ', r['max_abs_gap_unmasked'])
print('causal mask, max |gap|', r['max_abs_gap_causal'])

permutation            [3, 1, 4, 2]
seed used              20260730
no mask, max |gap|     9.5367431640625e-07
causal mask, max |gap| 6.4887


### Result 8.5, traced: mask before the softmax, not after

In [11]:
def claim_8_5_mask(n=4):
    """Masking before the softmax, and the same masking after it.

    Warrant traced. The claim is an ordering claim, so the instance has to be
    the two orderings side by side: mask then normalise leaves every row summing
    to one, normalise then mask leaves rows summing to i/n and is a different
    operation that no error message would ever report.
    """
    torch.manual_seed(SEED)
    S = torch.randn(n, n)
    causal = torch.tril(torch.ones(n, n, dtype=torch.bool))

    before = torch.softmax(S.masked_fill(~causal, float("-inf")), dim=-1)
    after = torch.softmax(S, dim=-1).masked_fill(~causal, 0.0)

    sums_before = [round(float(x), 6) for x in before.sum(-1)]
    sums_after = [round(float(x), 4) for x in after.sum(-1)]

    return {
        "n": n,
        "mask_lower_triangular": causal.int().tolist(),
        "allowed_per_row": [int(x) for x in causal.sum(-1)],
        "scores": [[round(float(x), 3) for x in r] for r in S],
        "weights_masked_before_softmax": [[round(float(x), 4) for x in r]
                                          for r in before],
        "weights_masked_after_softmax": [[round(float(x), 4) for x in r]
                                         for r in after],
        "row_sums_masked_before": sums_before,
        "row_sums_masked_after": sums_after,
        "max_row_sum_deficit_after": round(1 - min(sums_after), 4),
        "_ok": (max(abs(s - 1) for s in sums_before) < 1e-6
                and min(sums_after) < 0.999),
    }

r = claim_8_5_mask()
r.pop('_ok', None)

print('row sums, masked before softmax:', r['row_sums_masked_before'])
print('row sums, masked after softmax :', r['row_sums_masked_after'])

row sums, masked before softmax: [1.0, 1.0, 1.0, 1.0]
row sums, masked after softmax : [0.1968, 0.638, 0.5251, 1.0]


### Result 8.6, derived: h heads of width d/h cost what one head of width d costs

In [12]:
def claim_8_6_head_cost():
    """h heads of width d/h cost exactly what one head of width d costs.

    Warrant derived, and derived claims here get counted rather than asserted:
    both sublayers are built out of real torch modules and their parameters are
    counted, so the slide's 4d^2 is a measurement of an object and not an
    identity someone believed. The check frame asks the room to predict this,
    which makes a wrong number on it expensive.
    """
    def projections(h):
        d_k = D_MODEL // h
        #% bias=False deliberately: the slide's 4d^2 is a count of weights, and
        #% the Transformer's attention projections conventionally carry no bias.
        mods = [torch.nn.Linear(D_MODEL, h * d_k, bias=False) for _ in range(3)]
        mods.append(torch.nn.Linear(D_MODEL, D_MODEL, bias=False))
        return sum(p.numel() for m in mods for p in m.parameters()), d_k

    one, d_k_one = projections(1)
    many, d_k_many = projections(N_HEADS)
    qkv_many = 3 * N_HEADS * D_MODEL * d_k_many
    w_o = D_MODEL ** 2

    return {
        "d_model": D_MODEL, "h": N_HEADS,
        "d_k_single_head": d_k_one, "d_k_multi_head": d_k_many,
        "params_single_head": one, "params_multi_head": many,
        "params_single_head_tex": tex_int(one),
        "params_equal": one == many,
        "predicted_4d2": 4 * D_MODEL ** 2,
        "qkv_params": qkv_many, "output_projection_params": w_o,
        "params_multi_head_tex": tex_int(many),
        "qkv_params_tex": tex_int(qkv_many),
        "output_projection_params_tex": tex_int(w_o),
        #% Equal in parameters is not equal in anything else, which is the
        #% boundary the claim states. Give the room the number: eight heads run
        #% eight softmaxes over 64 dimensions, not one over 512.
        "softmax_width_single": D_MODEL, "softmax_width_multi": d_k_many,
        "_ok": one == many == 4 * D_MODEL ** 2 and qkv_many == 3 * D_MODEL ** 2,
    }

r = claim_8_6_head_cost()
r.pop('_ok', None)

print('one head of width', r['d_k_single_head'], ':', r['params_single_head'], 'parameters')
print(r['h'], 'heads of width', r['d_k_multi_head'], ':', r['params_multi_head'], 'parameters')
print('equal:', r['params_equal'], '  and 4d^2 =', r['predicted_4d2'])

one head of width 512 : 1048576 parameters
8 heads of width 64 : 1048576 parameters
equal: True   and 4d^2 = 1048576


### Result 8.9, derived: the residual gives the gradient an identity path

In [13]:
def claim_8_9_residual():
    """The residual gives the gradient an identity path: dy/dx = I + J.

    Two instances, because the claim has two halves. The identity is checked
    exactly, against autograd, so "I plus the sublayer's Jacobian" is verified
    rather than asserted. Then the consequence is measured through twenty-four
    sublayers with deliberately small weights: without the residual the gradient
    norm collapses by orders of magnitude, with it the identity path holds it up.
    The boundary is measured too, and it is the honest half: the identity path
    stops the collapse, it does nothing about growth.
    """
    torch.manual_seed(SEED)
    d, depth = 8, 24
    sub = torch.nn.Linear(d, d, bias=False)
    #% Small weights on purpose: this is the vanishing regime, which is the
    #% regime the claim is about.
    with torch.no_grad():
        sub.weight.mul_(0.3)

    x = torch.randn(d, requires_grad=True)
    J_f = torch.autograd.functional.jacobian(lambda v: torch.tanh(sub(v)), x)
    J_y = torch.autograd.functional.jacobian(
        lambda v: v + torch.tanh(sub(v)), x)
    identity_gap = float((J_y - (torch.eye(d) + J_f)).abs().max())

    def grad_norm(residual):
        torch.manual_seed(SEED)
        v = torch.randn(d, requires_grad=True)
        h = v
        for _ in range(depth):
            h = h + torch.tanh(sub(h)) if residual else torch.tanh(sub(h))
        h.sum().backward()
        return float(v.grad.norm())

    with_res, without_res = grad_norm(True), grad_norm(False)
    #% Growth is a statement about depth, so it is measured against depth and
    #% not against the tanh stack. Comparing 97 to 18.8 compares two different
    #% experiments and was the second wrong assertion this function has held.
    by_depth = {k: float("%.4g" % _grad_norm_linear_residual(d, k, 0.5))
                for k in (1, 8, 16, 24)}

    return {
        "d": d, "depth": depth, "sublayer_weight_scale": 0.3,
        "jacobian_identity_gap": identity_gap,
        "grad_norm_with_residual": round(with_res, 6),
        "grad_norm_without_residual": float("%.3g" % without_res),
        "ratio": float("%.4g" % (with_res / without_res)),
        #% The boundary, measured rather than asserted, and it took two attempts
        #% to measure honestly. The first version claimed that the same stack at
        #% weight scale 3.0 explodes; it gives 19.9 against 18.8, which is not
        #% growth and would have been a false statement generated by the very
        #% tool that exists to stop them. The reason is that a saturated tanh has
        #% a near-zero derivative, so at large weights I + J collapses *to* I and
        #% the identity path carries everything. To show growth the sublayer has
        #% to stay linear, and then I + W compounds through depth as it should.
        "note_boundary": ("the identity path prevents collapse, not growth. "
                          "With a saturating sublayer, large weights make J "
                          "vanish and the norm merely stops changing; with a "
                          "linear sublayer at scale 0.5 it compounds through "
                          "depth, which is what the normalisation is for."),
        "grad_norm_linear_residual_by_depth": by_depth,
        "growth_factor_1_to_24": float("%.3g" % (by_depth[24] / by_depth[1])),
        "_ok": (identity_gap < 1e-5 and with_res > 100 * without_res
                and by_depth[24] > 10 * by_depth[1]),
    }

r = claim_8_9_residual()
r.pop('_ok', None)

print('max |dy/dx - (I + J)| =', r['jacobian_identity_gap'])
print('through', r['depth'], 'sublayers:')
print('  with the residual   ', r['grad_norm_with_residual'])
print('  without it          ', r['grad_norm_without_residual'])
print('and the boundary, a linear residual stack by depth:')
print(' ', r['grad_norm_linear_residual_by_depth'])

max |dy/dx - (I + J)| = 0.0
through 24 sublayers:
  with the residual    18.807438
  without it           2.69e-18
and the boundary, a linear residual stack by depth:
  {1: 3.139, 8: 11.0, 16: 34.04, 24: 97.08}


### Result 8.11, derived: the feed-forward sublayer holds two thirds of the block

In [14]:
def claim_8_11_ffn_share():
    """The feed-forward sublayer holds two thirds of the block's parameters.

    Counted off real modules, like 8.6. And the count is reported twice, because
    the slide quotes 2 x 512 x 2048 = 2{,}097{,}152, which is weights only. With
    biases and the two layer norms the block is slightly larger and the share is
    slightly different, and a claim about a ratio should say which ratio it means.
    """
    attn = 4 * D_MODEL ** 2
    ffn_w = torch.nn.Sequential(torch.nn.Linear(D_MODEL, D_FF, bias=False),
                                torch.nn.Linear(D_FF, D_MODEL, bias=False))
    ffn = sum(p.numel() for p in ffn_w.parameters())

    ffn_b = torch.nn.Sequential(torch.nn.Linear(D_MODEL, D_FF),
                                torch.nn.Linear(D_FF, D_MODEL))
    ffn_with_bias = sum(p.numel() for p in ffn_b.parameters())
    #% Two layer norms, gain and bias each.
    norms = 4 * D_MODEL
    total_full = attn + ffn_with_bias + norms

    return {
        "d_model": D_MODEL, "d_ff": D_FF, "d_ff_multiple": D_FF // D_MODEL,
        "attention_params": attn, "attention_params_tex": tex_int(attn),
        "ffn_params": ffn, "ffn_params_tex": tex_int(ffn),
        "block_params": attn + ffn, "block_params_tex": tex_int(attn + ffn),
        "ffn_share": round(ffn / (attn + ffn), 4),
        "ffn_share_as_fraction": "8d^2 of 12d^2",
        "ffn_params_with_bias": ffn_with_bias,
        "layernorm_params": norms,
        "block_params_with_bias_and_norms": total_full,
        "ffn_share_with_bias_and_norms": round(ffn_with_bias / total_full, 4),
        "_ok": (ffn == 2 * D_MODEL * D_FF
                and abs(ffn / (attn + ffn) - 2 / 3) < 1e-9),
    }

r = claim_8_11_ffn_share()
r.pop('_ok', None)

print('attention   ', r['attention_params'])
print('feed-forward', r['ffn_params'])
print('share       ', r['ffn_share'], '=', r['ffn_share_as_fraction'])

attention    1048576
feed-forward 2097152
share        0.6667 = 8d^2 of 12d^2
